In [1]:
import os
import re
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [14]:
class SGLX_readMeta:
    """Python port of SGLX_readMeta.m. Method names/semantics preserved."""

    # -----------------------------------------------------------------
    @staticmethod
    def ReadMeta(binName, path):
        """Parse the matching .meta file into a dict of strings."""
        # MATLAB fileparts() strips only the final extension:
        #   'foo.nidq.bin' -> name='foo.nidq'
        name = os.path.splitext(os.path.basename(binName))[0]
        metaName = name + ".meta"
        metaPath = Path(path) / metaName

        meta = {}
        with metaPath.open("r") as f:
            for line in f.read().splitlines():
                if "=" not in line:
                    continue
                tag, _, val = line.partition("=")
                if tag.startswith("~"):
                    tag = tag[1:]
                meta[tag] = val
        return meta

    # -----------------------------------------------------------------
    @staticmethod
    def ChannelCountsIM(meta):
        M = _matlab_str2num_list(meta["snsApLfSy"])
        return int(M[0]), int(M[1]), int(M[2])

    @staticmethod
    def ChannelCountsNI(meta):
        M = _matlab_str2num_list(meta["snsMnMaXaDw"])
        return int(M[0]), int(M[1]), int(M[2]), int(M[3])

    @staticmethod
    def ChannelCountsOBX(meta):
        M = _matlab_str2num_list(meta["snsXaDwSy"])
        return int(M[0]), int(M[1]), int(M[2])

def _matlab_str2num_list(s):
    """Parse a MATLAB str2num-style numeric list, e.g. '384,0,1' or
    '0:33,36:37,40' into a flat list of ints/floats."""
    vals = []
    for part in s.split(","):
        part = part.strip()
        if not part:
            continue
        if ":" in part:
            lo, hi = part.split(":")
            vals.extend(range(int(lo), int(hi) + 1))
        else:
            vals.append(float(part) if "." in part else int(part))
    return vals

In [2]:
def read_cluster_kslabel_tsv(tsv_path):
    return pd.read_csv(
        tsv_path, sep="\t", header=0, names=["cluster_id", "KSLabel"],
        dtype={"cluster_id": float, "KSLabel": str},
    )

### getData: start from step01_rawData

In [15]:
# Windows absolute path -- use a raw string (r"...") so backslashes
# aren't treated as escape characters, or use forward slashes instead.
folderPath = r"C:\Users\mymcm\OneDrive - Northwestern University\Hartmann Lab - Mitra Hartmann - expt_260605_rat17"

assert os.path.isdir(folderPath), f"Folder not found: {folderPath}"

allFiles = glob.glob(os.path.join(folderPath, "**", "*"), recursive=True)
binNidqFiles = [f for f in allFiles if f.endswith(".nidq.bin")]
binImecFiles = [f for f in allFiles if f.endswith(".imec0.ap.bin")]
csvFiles = [f for f in allFiles if f.endswith(".csv")]

print(f"Found {len(binNidqFiles)} nidq.bin, {len(binImecFiles)} imec0.ap.bin, {len(csvFiles)} csv files")

print("\nbinNidqFiles:")
for idx, f in enumerate(binNidqFiles):
    print(f"[{idx + 1}] {f}")

Found 18 nidq.bin, 10 imec0.ap.bin, 31 csv files

binNidqFiles:
[1] C:\Users\mymcm\OneDrive - Northwestern University\Hartmann Lab - Mitra Hartmann - expt_260605_rat17\rat17_ap3000_ml4600_dv1320_g0\rat17_ap3000_ml4600_dv1320_g0_t0.nidq.bin
[2] C:\Users\mymcm\OneDrive - Northwestern University\Hartmann Lab - Mitra Hartmann - expt_260605_rat17\rat17_ap3000_ml4600_dv1600_02_g0\rat17_ap3000_ml4600_dv1600_02_g0_t0.nidq.bin
[3] C:\Users\mymcm\OneDrive - Northwestern University\Hartmann Lab - Mitra Hartmann - expt_260605_rat17\rat17_ap3000_ml4600_dv1600_02_g0\copy\rat17_ap3000_ml4600_dv1600_02_g0_t0.nidq.bin
[4] C:\Users\mymcm\OneDrive - Northwestern University\Hartmann Lab - Mitra Hartmann - expt_260605_rat17\rat17_ap3000_ml4600_dv1600_B4_g0\rat17_ap3000_ml4600_dv1600_B4_g0_t0.nidq.bin
[5] C:\Users\mymcm\OneDrive - Northwestern University\Hartmann Lab - Mitra Hartmann - expt_260605_rat17\rat17_ap3000_ml4600_dv1600_g0\rat17_ap3000_ml4600_dv1600_g0_t0.nidq.bin
[6] C:\Users\mymcm\OneDrive - Nor

In [16]:
j = int(input("which neural data number do you want to analyze? ")) - 1  # MATLAB is 1-indexed
imecFilePath = binImecFiles[j]
binPath = os.path.dirname(imecFilePath)
binNameFull = os.path.basename(imecFilePath)
binName = binNameFull[: -len(".imec0.ap.bin")]
print(binName)
print(binPath)

# 1) Read metadata
imecBinName = binName + ".imec0.ap.bin"
imecMeta = SGLX_readMeta.ReadMeta(imecBinName, binPath)

nidqBinName = binName + ".nidq.bin"
nidqParentPath = os.path.dirname(binPath)  # fileparts(binPath)
nidqMeta = SGLX_readMeta.ReadMeta(nidqBinName, nidqParentPath)

rec_time = float(imecMeta["fileTimeSecs"])  # imec/nidq timesecs differ slightly, ignored
print(rec_time)

which neural data number do you want to analyze?  2


rat17_ap3000_ml4600_dv1600_02_g0_t0
C:\Users\mymcm\OneDrive - Northwestern University\Hartmann Lab - Mitra Hartmann - expt_260605_rat17\rat17_ap3000_ml4600_dv1600_02_g0\rat17_ap3000_ml4600_dv1600_02_g0_imec0
910.8197


In [17]:
ksDir = os.path.join(binPath, "Kilosort4")
spike_times = np.load(os.path.join(ksDir, "spike_times.npy"))       # sample index per spike
spike_clusters = np.load(os.path.join(ksDir, "spike_clusters.npy")) # cluster/template id per spike
#462xnearest_chans
pc_feature_ind = np.load(os.path.join(ksDir, "pc_feature_ind.npy")) # Channel indices of nearest channels for each template
#462x61x384
templates = np.load(os.path.join(ksDir, "templates.npy"))           # Full time x channels template shapes
templates_ind = np.load(os.path.join(ksDir, "templates_ind.npy"))
#len = n_channels
channel_positions = np.load(os.path.join(ksDir, "channel_positions.npy")) # x- and y- positions of probe contacts in microns
channel_shanks = np.load(os.path.join(ksDir, "channel_shanks.npy")) # shank index for each channel on the probe
channel_map = np.load(os.path.join(ksDir, "channel_map.npy"))       # probe[‘chanMap’]

In [18]:
imec_fs = float(imecMeta["imSampRate"])
imec_AP, imec_LF, imec_SY = SGLX_readMeta.ChannelCountsIM(imecMeta)
spike_sec = spike_times.astype(float) / imec_fs                      # -> seconds

# Prefer bombcell's unit classification (cluster_bc_unitType.tsv, label
# 'GOOD'); fall back to Kilosort's own cluster_KSLabel.tsv (label 'good')
# if bombcell output isn't present.
bcFile = os.path.join(ksDir, "cluster_bc_unitType.tsv")
ksFile = os.path.join(ksDir, "cluster_KSLabel.tsv")

if os.path.exists(bcFile):
    clusterLabels = pd.read_csv(bcFile, sep="\t")
    goodClusters = clusterLabels.loc[clusterLabels["bc_unitType"] == "GOOD", "cluster_id"].to_numpy()
    print(f"Using bombcell labels ({os.path.basename(bcFile)}): {len(goodClusters)} good units")
    print(f"Total clusters: {len(clusterLabels)} clusters")
else:
    clusterLabels = read_cluster_kslabel_tsv(ksFile)
    goodClusters = clusterLabels.loc[clusterLabels["KSLabel"] == "good", "cluster_id"].to_numpy()
    print(f"bombcell labels not found -- using Kilosort labels ({os.path.basename(ksFile)}): {len(goodClusters)} good units")

idxGoodSpike = np.isin(spike_clusters.astype(float), goodClusters)
spike_sec = spike_sec[idxGoodSpike]
goodClusterIDs = np.sort(goodClusters)

print(f"{len(goodClusterIDs)} good units, {len(spike_times)} total spikes")

Using bombcell labels (cluster_bc_unitType.tsv): 16 good units
Total clusters: 462 clusters
16 good units, 1267155 total spikes


In [19]:
#make an array where each good cluster ID is tied to its channels
    #each cluster ID (clusterLabels->goodClusters) corresponds to a template
    #each template has some nearby channels (pc_feature_ind)
goodChannels = np.asarray([pc_feature_ind[i] for i in goodClusterIDs])

#x axis - identifies which of the 8 columns an electrode is on (2 columns on each of 4 shanks)
#y axis - depth along the probe (0 at tip)
#make a table where the good cluster channels are mapped to the channel locations in space (channel_positions)
d = []
for ID, cluster in zip(goodClusterIDs, goodChannels):
    d.append(
        {
            'Cluster ID': ID,
            'Shank': channel_shanks[cluster[0]], #the shank for all channels in a cluster is the same
            'Pos1': channel_positions[cluster[0]],
            'Pos2': channel_positions[cluster[1]],
            'Pos3': channel_positions[cluster[2]],
            'Pos4': channel_positions[cluster[3]],
            'Pos5': channel_positions[cluster[4]],
            'Pos6': channel_positions[cluster[5]],
            'Pos7': channel_positions[cluster[6]],
            'Pos8': channel_positions[cluster[7]],
            'Pos9': channel_positions[cluster[8]],
            'Pos10': channel_positions[cluster[9]]
        }
    )

pd.DataFrame(d)

,Cluster ID,Shank,Pos1,Pos2,Pos3,Pos4,Pos5,Pos6,Pos7,Pos8,Pos9,Pos10
0,0,1.0,"[27.0, 0.0]","[27.0, 15.0]","[27.0, 30.0]","[27.0, 45.0]","[59.0, 30.0]","[59.0, 45.0]","[59.0, 15.0]","[59.0, 60.0]","[27.0, 60.0]","[59.0, 0.0]"
1,86,2.0,"[277.0, 0.0]","[277.0, 15.0]","[277.0, 30.0]","[309.0, 30.0]","[309.0, 0.0]","[309.0, 15.0]","[277.0, 60.0]","[309.0, 60.0]","[309.0, 45.0]","[277.0, 45.0]"
2,208,3.0,"[527.0, 45.0]","[527.0, 30.0]","[527.0, 75.0]","[527.0, 15.0]","[527.0, 60.0]","[559.0, 45.0]","[559.0, 60.0]","[559.0, 30.0]","[559.0, 75.0]","[559.0, 15.0]"
3,217,3.0,"[559.0, 60.0]","[559.0, 75.0]","[559.0, 90.0]","[527.0, 75.0]","[559.0, 45.0]","[527.0, 60.0]","[559.0, 105.0]","[527.0, 90.0]","[527.0, 45.0]","[527.0, 30.0]"
4,233,3.0,"[559.0, 165.0]","[559.0, 150.0]","[527.0, 180.0]","[559.0, 180.0]","[527.0, 150.0]","[559.0, 135.0]","[527.0, 165.0]","[527.0, 195.0]","[559.0, 120.0]","[527.0, 135.0]"
5,269,3.0,"[527.0, 315.0]","[527.0, 285.0]","[527.0, 300.0]","[559.0, 285.0]","[559.0, 270.0]","[559.0, 330.0]","[559.0, 300.0]","[527.0, 330.0]","[559.0, 315.0]","[559.0, 345.0]"
6,309,3.0,"[559.0, 585.0]","[527.0, 540.0]","[527.0, 585.0]","[559.0, 570.0]","[559.0, 540.0]","[559.0, 525.0]","[527.0, 525.0]","[527.0, 555.0]","[559.0, 555.0]","[527.0, 570.0]"
7,338,4.0,"[777.0, 45.0]","[777.0, 30.0]","[777.0, 15.0]","[777.0, 60.0]","[809.0, 45.0]","[777.0, 75.0]","[809.0, 60.0]","[809.0, 15.0]","[809.0, 30.0]","[809.0, 0.0]"
8,346,4.0,"[809.0, 105.0]","[809.0, 90.0]","[809.0, 120.0]","[777.0, 105.0]","[777.0, 135.0]","[809.0, 75.0]","[777.0, 90.0]","[777.0, 75.0]","[777.0, 120.0]","[809.0, 135.0]"
9,347,4.0,"[809.0, 105.0]","[809.0, 90.0]","[809.0, 120.0]","[809.0, 135.0]","[809.0, 75.0]","[777.0, 75.0]","[777.0, 90.0]","[777.0, 105.0]","[777.0, 135.0]","[777.0, 120.0]"
